# Kafka et Spark Structured Streaming

Lecture des transactions Kafka, calcul du revenu et attribution du segment client à partir de la segmentation enregistrée.

## 1. Initialisation de Spark

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("EcommerceCustomerStreaming")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0"
    )
    .getOrCreate()
)

## 2. Lecture du topic Kafka

In [2]:
stream_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "customer-transactions")
    .option("startingOffsets", "latest")
    .load()
)

## 3. Parsing et calcul du revenu

In [3]:
schema = StructType([
    StructField("CustomerID", IntegerType()),
    StructField("InvoiceNo", StringType()),
    StructField("Quantity", IntegerType()),
    StructField("UnitPrice", DoubleType())
])

transactions = (
    stream_df
    .select(
        F.from_json(
            F.col("value").cast("string"),
            schema
        ).alias("data")
    )
    .select("data.*")
    .filter(
        F.col("CustomerID").isNotNull()
        & F.col("InvoiceNo").isNotNull()
        & F.col("Quantity").isNotNull()
        & F.col("UnitPrice").isNotNull()
    )
    .withColumn("Revenue", F.col("Quantity") * F.col("UnitPrice"))
)

## 4. Chargement de la segmentation existante

In [4]:
customer_segments = spark.read.parquet(
    "/home/jovyan/work/data/processed/customer_segments"
)

stream_segmented = transactions.join(
    customer_segments.select("CustomerID", "Cluster"),
    on="CustomerID",
    how="left"
)

## 5. Démarrage du streaming

In [5]:
query_segmented = (
    stream_segmented
    .writeStream
    .format("memory")
    .queryName("live_customer_segments")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "/home/jovyan/work/data/streaming/checkpoint_final"
    )
    .start()
)

print("Streaming active:", query_segmented.isActive)

Streaming active: True


## 6. Consultation des transactions traitées

- To send demo data use this command:

  ```bash
  docker exec -it customer-segmentation-kafka \       
  /opt/kafka/bin/kafka-console-producer.sh \
  --topic customer-transactions \
  --bootstrap-server localhost:9092
  ```

  - Send commands for example:
      - `{"CustomerID":12748,"InvoiceNo": "LIVE_FINAL","Quantity":4,"UnitPrice":25.0}`

In [11]:
query_segmented.processAllAvailable()

spark.sql("""
    SELECT CustomerID, InvoiceNo, Quantity, UnitPrice, Revenue, Cluster
    FROM live_customer_segments
    ORDER BY InvoiceNo
""").show(truncate=False)

+----------+----------+--------+---------+---------+-------+
|CustomerID|InvoiceNo |Quantity|UnitPrice|Revenue  |Cluster|
+----------+----------+--------+---------+---------+-------+
|12748     |LIVE_FINAL|4       |25.0     |100.0    |0      |
|12748     |fdsf      |1122    |5123.0   |5748006.0|0      |
|12748     |test2     |1       |5.0      |5.0      |0      |
+----------+----------+--------+---------+---------+-------+



## 7. Arrêt du streaming

In [12]:
query_segmented.stop()